## Environment Setup

In [ ]:
!pip uninstall -y -q tensorflow
!pip install -q --upgrade pip
!pip install -q transformers==4.40.0 accelerate==0.29.3 datasets==2.19.0 jiwer soundfile librosa

import os
import glob
import pandas as pd
import torch
from transformers import pipeline
from tqdm.notebook import tqdm

device = 0 if torch.cuda.is_available() else -1
print(f"Device set to: {'GPU' if device == 0 else 'CPU'}")

In [ ]:
MODEL_ID = "anvitamanne/whisper-train
pipe = pipeline(
    "automatic-speech-recognition", 
    model=MODEL_ID, 
    device=device 
)
print("Model loaded successfully.")

## Mapping Noisy Dataset

In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
!mv /workspace/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Downloading curated noise-augmented Telugu speech corpus
!kaggle datasets download -d anvitamanne/telugu-noisy-data --unzip -p /workspace/data
!apt-get update && apt-get install -y ffmpeg

In [ ]:
BASE_DIR = "/workspace/data"
MAPPING_CONFIG = {
    "mozilla": {"csv": os.path.join(BASE_DIR, "archive (1)/mozilla_transcripts_csv/transcripts.csv"), "audio_dir": os.path.join(BASE_DIR, "archive (1)/mozilla_noise")},
    "indictts": {"csv": os.path.join(BASE_DIR, "archive (2)/indictts_noise_transcripts.csv"), "audio_dir": os.path.join(BASE_DIR, "archive (2)/indictts_noise_audio")},
    "openslr": {"csv": os.path.join(BASE_DIR, "archive (3)/noisy_dataset/transcriptions.csv"), "audio_dir": os.path.join(BASE_DIR, "archive (3)/noisy_dataset/clips")}
}

master_data = []
for label, paths in MAPPING_CONFIG.items():
    if os.path.exists(paths['csv']):
        df = pd.read_csv(paths['csv'])
        df.columns = [c.lower().strip() for c in df.columns]
        file_col = next((c for c in df.columns if any(x in c for x in ['file', 'path', 'audio'])), None)
        text_col = next((c for c in df.columns if any(x in c for x in ['text', 'sentence', 'transcript'])), None)
        
        for _, row in df.iterrows():
            fname = os.path.basename(str(row[file_col]))
            full_audio_path = os.path.join(paths['audio_dir'], fname)
            if os.path.exists(full_audio_path):
                master_data.append({"dataset": label, "audio_path": full_audio_path, "reference": str(row[text_col])})

final_df = pd.DataFrame(master_data)
print(f"Success! Found {len(final_df)} valid audio-text pairs.")

## Performance Results

In [ ]:
from jiwer import wer, cer
import re

def normalize_telugu(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[^\u0C00-\u0C7F\s]', '', text) 
    return re.sub(r'\s+', ' ', text).strip()

individual_results = []
for idx, row in tqdm(final_df.iterrows(), total=len(final_df)):
    try:
        out = pipe(row['audio_path'], generate_kwargs={"language": "telugu"})
        ref_clean = normalize_telugu(row['reference'])
        hyp_clean = normalize_telugu(out["text"])
        individual_results.append({
            "dataset": row['dataset'], "reference": ref_clean, "prediction": hyp_clean,
            "wer": wer(ref_clean, hyp_clean) if ref_clean else 0,
            "cer": cer(ref_clean, hyp_clean) if ref_clean else 0
        })
    except Exception: continue

res_df = pd.DataFrame(individual_results)

In [ ]:
all_refs, all_hyps = res_df['reference'].tolist(), res_df['prediction'].tolist()
total_corpus_wer = wer(all_refs, all_hyps)
total_corpus_cer = cer(all_refs, all_hyps)

summary_df = pd.DataFrame({
    "Total Files Processed": [len(res_df)],
    "Global Corpus WER": [f"{total_corpus_wer:.4f} ({total_corpus_wer*100:.2f}%)"],
    "Global Corpus CER": [f"{total_corpus_cer:.4f} ({total_corpus_cer*100:.2f}%)"]
})

display(summary_df)
res_df.to_csv("whisper_full_corpus_results.csv", index=False)